# Adaptive Business Management Tutor
### Multi-Agent System with LangChain ReAct

**Architecture:** A supervisor agent receives the user's request, decides which sub-agent to invoke, and each sub-agent has its own set of tools it can call autonomously.

```
User Input
    │
    ▼
Supervisor Agent  ──── decides ────►  Quick Study Agent    (tools: retrieve, summarise, gen_quiz, export_pdf)
                                  └►  Full Scale Agent     (orchestrates 4 sub-agents below)
                                            │
                                            ├─ Knowledge Assessment Agent  (tools: retrieve, gen_quiz, run_quiz, save_result)
                                            ├─ Learning Path Agent         (tools: fetch_results, analyse_gaps, save_path)
                                            ├─ Adaptive Quiz Agent         (tools: gen_quiz, run_quiz, score, check_pass)
                                            └─ Mastery Tracking Agent      (tools: update_mastery, gen_summary, export_pdf)
```

**Stack:** LangChain · Groq (llama3-8b-8192) · ChromaDB · SQLite · ReportLab

---
Run cells top to bottom. Cell 6 is the interactive entry point.

## Cell 1 — Install dependencies

In [2]:
import subprocess, sys

# Uninstall old incompatible versions
# subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "langchain-groq", "groq", "-y", "-q"])

packages = [
    # "langchain==0.3.25",
    # "langchain-core==0.3.86",
    # "langchain-community==0.3.24",
    # "langchain-groq==0.3.4",
    # "langchain-text-splitters==0.3.8",
    # "pydantic==2.11.4",
    # "groq==0.13.0",
    "python-dotenv",
    "sentence-transformers",
    "chromadb",
    "reportlab",
    "pypdf",
    "datasets",
    "pandas",
]
for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p])

print("✓ All packages installed with compatible versions.")

You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


✓ All packages installed with compatible versions.


You should consider upgrading via the '/Users/akankshacheeti/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.


In [ ]:
# !pip install langgraph>=0.3.1
# !pip show langgraph

In [3]:
!pip show langchain_groq

Name: langchain-groq
Version: 1.1.3
Summary: An integration package connecting Groq and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /Users/akankshacheeti/.pyenv/versions/3.10.4/lib/python3.10/site-packages
Requires: groq, langchain-core
Required-by: 


## Cell 2 — Imports & configuration

In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent


/Users/akankshacheeti/.pyenv/versions/3.10.4/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# %pip install --upgrade huggingface_hub

In [6]:
# %pip install --upgrade transformers sentence-transformers numpy
from sentence_transformers import SentenceTransformer

In [10]:
#Note: import dependencies and load environment variables like groq api key, chromadb path, etc. from .env file

import os, json, sqlite3, uuid, textwrap
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# LangChain
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent
# from langchainhub import hub

# Vector DB & embeddings
from sentence_transformers import SentenceTransformer
import chromadb

# PDF
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

load_dotenv()

True

In [11]:
# ── Config ──────────────────────────────────────────────────────────────────
GROQ_MODEL      = "llama-3.3-70b-versatile"
EMBED_MODEL     = "all-MiniLM-L6-v2"
CHROMA_PATH     = "./chroma_db"
CHROMA_SCIQ_COLLECTION = "sciq_knowledge_base"
COLLECTION_NAME = "knowledge_base"
DB_PATH         = "./tutor.db"
OUTPUT_DIR      = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
PASS_THRESHOLD  = 0.80
MAX_QUIZ_ROUNDS = 4


In [12]:
# ── Shared clients ───────────────────────────────────────────────────────────
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.2,
    max_tokens=2000,
)


In [13]:
#Note: Test if the llm call works
llm.invoke("Hello, world!")

AIMessage(content="Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 39, 'total_tokens': 64, 'completion_time': 0.049722881, 'completion_tokens_details': None, 'prompt_time': 0.0036914, 'prompt_tokens_details': None, 'queue_time': 0.16360449, 'total_time': 0.053414281}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec1fc-289f-7202-83ad-c947e63572a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 25, 'total_tokens': 64})

In [14]:
# import os
# os.environ["CHROMADB_TELEMETRY_DISABLED"] = "true"

from sentence_transformers import SentenceTransformer
import chromadb

embed_model   = SentenceTransformer(EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection    = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
collection_sciq = chroma_client.get_or_create_collection(name=CHROMA_SCIQ_COLLECTION)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6835.11it/s]


In [15]:
#Suppress warnings from langchain_groq and pydantic
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_groq")


import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
import os

# 1. Block Python's standard warning display
warnings.filterwarnings("ignore")

# 2. Block Pydantic's specific deprecation messages
try:
    from pydantic import PydanticDeprecatedSince20
    warnings.filterwarnings("ignore", category=PydanticDeprecatedSince20)
except ImportError:
    pass

# 3. Tell the environment to stop Pydantic validator warnings
os.environ["PYDANTIC_SKIP_VALIDATOR_WARNINGS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"


In [ ]:
# ── Session state (single user) ──────────────────────────────────────────────
# This dict is passed between agents so they share context, context like the chunks retrieveded from chromadb, the diagnostic score, the adaptive result, etc.
SESSION = {
    "profile":          None,
    "context":          "",
    "diagnostic_score": None,
    "gap_data":         None,
    "adaptive_result":  None,
    "summary": None
}

print(f"LLM ready: {GROQ_MODEL}")
print(f"ChromaDB chunks: {collection.count()}") # FOR THE PDF 
print(f"SciQ chunks: {collection_sciq.count()}")

LLM ready: llama-3.3-70b-versatile
ChromaDB chunks: 0
SciQ chunks: 6269


## Cell 3 — Database schema

In [12]:
#note: Initialize the SQLite database with tables for learner profiles, quiz sessions, and mastery records.
def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS learner_profiles (
            id         TEXT PRIMARY KEY,
            name       TEXT NOT NULL,
            topic      TEXT NOT NULL,
            level      TEXT DEFAULT 'beginner',
            created_at TEXT DEFAULT (datetime('now'))
        );
        CREATE TABLE IF NOT EXISTS quiz_sessions (
            id             TEXT PRIMARY KEY,
            learner_id     TEXT NOT NULL,
            session_type   TEXT NOT NULL,
            topic          TEXT NOT NULL,
            score_pct      REAL,
            passed         INTEGER DEFAULT 0,
            round_number   INTEGER DEFAULT 1,
            responses_json TEXT,
            created_at     TEXT DEFAULT (datetime('now'))
        );
        CREATE TABLE IF NOT EXISTS mastery_records (
            id         TEXT PRIMARY KEY,
            learner_id TEXT NOT NULL,
            topic      TEXT NOT NULL,
            mastered   INTEGER DEFAULT 0,
            best_score REAL DEFAULT 0.0,
            updated_at TEXT DEFAULT (datetime('now'))
        );
    """)
    conn.commit()
    conn.close()

init_db()
print("Database ready:", DB_PATH)

Database ready: ./tutor.db


## Cell 4 — Tool functions

Every tool is a plain Python function first, then wrapped with `langchain_core.tools.Tool`.
Agents call these by name — the LLM decides *which* tool to use and *what* to pass to it.

In [13]:
# ── 1. Profile tools ────────────────────────────────────────────────────────

def create_learner_profile(input_str: str) -> str:
    """
    Input: "name|topic|level"  e.g. "Alice|Financial Statements|beginner"
    Creates a learner profile in SQLite. Returns the profile as JSON string.
    """
    parts = [p.strip() for p in input_str.split("|")]
    name  = parts[0] if len(parts) > 0 else "Learner"
    topic = parts[1] if len(parts) > 1 else "Business Management"
    level = parts[2] if len(parts) > 2 else "beginner"
    if level not in ("beginner", "intermediate", "advanced"):
        level = "beginner"

    learner_id = str(uuid.uuid4())[:8]
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO learner_profiles (id,name,topic,level) VALUES (?,?,?,?)",
        (learner_id, name, topic, level)
    )
    conn.commit(); conn.close()

    profile = {"id": learner_id, "name": name, "topic": topic, "level": level}
    SESSION["profile"] = profile
    return json.dumps(profile)


def fetch_learner_profile(learner_id: str) -> str: # Note: Actually we are not using it yet, but we will in the future to prevent duplicate profiles from being added)
    """
    Input: learner_id string.
    Returns the learner profile as a JSON string, or an error message.
    """
    conn = sqlite3.connect(DB_PATH)
    row  = conn.execute(
        "SELECT id,name,topic,level FROM learner_profiles WHERE id=?",
        (learner_id.strip(),)
    ).fetchone()
    conn.close()
    if row:
        profile = dict(zip(["id","name","topic","level"], row))
        SESSION["profile"] = profile
        return json.dumps(profile)
    return f"No profile found for ID: {learner_id}"

In [160]:
# fetch_learner_profile("d5628551")

In [17]:
#Note: One other tool that is being used in the quick study agent is the retrieve_study_material function which retrieves the relevant chunks from chromadb based on the topic and stores it in the session context so that it can be used by other agents.
def retrieve_study_material_sciq(question: str) -> str:
    """
    Input: question string (e.g. "What is photosynthesis?").
    Retrieves the most relevant chunks from the SciQ collection in ChromaDB.
    Returns the joined context text.
    """
    question = question.strip()
    if collection_sciq.count() == 0:
        return f"No study material found in SciQ knowledge base for: {question}"
    emb  = embed_model.encode([question])[0].tolist()
    res  = collection_sciq.query(query_embeddings=[emb], n_results=min(15, collection_sciq.count()))
    ctx  = "\n\n".join(res["documents"][0])
    SESSION["context"] = ctx
    print(f"Retrieved {len(res['documents'][0])} chunks from SciQ for question '{question}'. Total context size: {len(ctx)} chars.")
    return ctx


def retrieve_study_material(topic: str) -> str:
    """
    Input: topic string (e.g. "Financial Statements").
    Retrieves the most relevant chunks from ChromaDB.
    Returns the joined context text.
    """
    topic = topic.strip()
    if collection.count() == 0:
        return f"No study material found in knowledge base for: {topic}"
    emb  = embed_model.encode([topic])[0].tolist()
    res  = collection.query(query_embeddings=[emb], n_results=min(8, collection.count()))
    ctx  = "\n\n".join(res["documents"][0])
    SESSION["context"] = ctx
    print(f"Retrieved {len(res['documents'][0])} chunks for topic '{topic}'. Total context size: {len(ctx)} chars.")
    return ctx

In [ ]:
# #Note: Test if the study material retrieval works
# returns = retrieve_study_material("Financial Statements")
# returns[:500]

Retrieved 15 chunks for topic 'Financial Statements'. Total context size: 15028 chars.


's: The users of financial statements include\nShareholders, Investors, Creditors, Lenders, Customers, Management, Government,\netc. Financial statements help all the users in their decision-making process. Theyprovide data about general purpose needs of these members.\nLimitations of Financial Statements: Financial statements are not free from\nlimitations. They provide only aggregate information to satisfy the general purpose\nneeds of the users. They are technical statements understood by only pers'

In [18]:
#returns_sciq = retrieve_study_material_sciq("What kind of a reaction occurs when a substance reacts quickly with oxygen?")
returns_sciq=retrieve_study_material_sciq("explain about intenstines")
returns_sciq

Retrieved 15 chunks from SciQ for question 'explain about intenstines'. Total context size: 9192 chars.


'Question: The ileum is the last part of what organ, and is where the bile salts and vitamins are absorbed into blood stream?\n\nSupport: The ileum, also illustrated in Figure 34.11 is the last part of the small intestine and here the bile salts and vitamins are absorbed into blood stream. The undigested food is sent to the colon from the ileum via peristaltic movements of the muscle. The ileum ends and the large intestine begins at the ileocecal valve. The vermiform, “worm-like,” appendix is located at the ileocecal valve. The appendix of humans secretes no enzymes and has an insignificant role in immunity. Large Intestine The large intestine, illustrated in Figure 34.13, reabsorbs the water from the undigested food material and processes the waste material. The human large intestine is much smaller in length compared to the small intestine but larger in diameter. It has three parts: the cecum, the colon, and the rectum. The cecum joins the ileum to the colon and is the receiving pouc

In [15]:
# ── 2. Retrieval tool ────────────────────────────────────────────────────────

#Note: This cell defines all the tools (you must run it to initialize the tools)
def retrieve_study_material(topic: str) -> str:
    """
    Input: topic string (e.g. "Financial Statements").
    Retrieves the most relevant chunks from ChromaDB.
    Returns the joined context text.
    """
    topic = topic.strip()
    if collection.count() == 0:
        return f"No study material found in knowledge base for: {topic}"
    emb  = embed_model.encode([topic])[0].tolist()
    res  = collection.query(query_embeddings=[emb], n_results=min(8, collection.count()))
    ctx  = "\n\n".join(res["documents"][0])
    SESSION["context"] = ctx
    print(f"Retrieved {len(res['documents'][0])} chunks for topic '{topic}'.")
    print(f"Context preview:\n{ctx[:500]}...")
    return ctx

# ── 9. Generate summary tool ──────────────────────────────────────────────────

def generate_study_summary(topic: str) -> str:
    """
    Input: topic string.
    Generates main points, overview summary, and key exam questions.
    Returns JSON string with keys: main_points, summary, key_questions.
    """
    topic = topic.strip()
    ctx   = SESSION.get("context") or retrieve_study_material(topic)

    profile = SESSION.get("profile", {"level":"beginner"})
    prompt  = f"""You are a business management tutor making a detailed study aid for a {profile['level']} student.
Topic: {topic}

Using ONLY the context below, produce:
1. main_points: 15-30 key takeaways as detailed bullet strings, be concise but comprehensive, covering all major concepts. Each point should be 1-3 sentences.
2. summary: comprehensive 2-6 paragraph overview covering the topic depth, based on the complexity of the material and learner level.
3. key_questions: 5-8 important exam-style questions the student should be able to answer, with hints if possible.

Be thorough and comprehensive. Cover all major concepts from the material.

Context:
{ctx[:8000]}

Return ONLY valid JSON — no markdown:
{{"main_points":["..."],"summary":"...","key_questions":["..."]}}"""

    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4,
        max_tokens=4000,
    )
    raw = (resp.choices[0].message.content or "").strip()
    s   = raw[raw.find("{"):raw.rfind("}")+1]
    try:
        print(s)
        SESSION["summary"] = json.loads(s)
        return s
        
    except Exception:
        return json.dumps({"main_points":[],"summary":f"Study notes for {topic}.","key_questions":[]})


# ── 10. Export PDF tool ───────────────────────────────────────────────────────

def export_report_pdf(report_type: str) -> str:
    """
    Input: "quick_study" or "mastery_report"
    Reads SESSION for profile, summary, quiz results, gaps.
    Builds and saves a PDF. Returns the file path.
    """
    report_type = report_type.strip().lower()
    profile     = SESSION.get("profile", {"name":"Learner","topic":"Unknown","level":"beginner","id":"0"})

    # Get summary data
    summary_raw  = generate_study_summary(profile["topic"])
    try:
        summary_data = json.loads(summary_raw)
    except Exception:
        summary_data = {"main_points":[],"summary":"","key_questions":[]}

    title    = "Quick Study Guide" if report_type == "quick_study" else "Mastery Report"
    filename = f"{report_type}_{profile['name'].replace(' ','_')}_{profile['topic'].replace(' ','_')[:20]}.pdf"
    filepath = OUTPUT_DIR / filename

    # ── Build PDF ────────────────────────────────────────────────────────────
    doc    = SimpleDocTemplate(str(filepath), pagesize=letter,
                               leftMargin=inch, rightMargin=inch,
                               topMargin=inch, bottomMargin=inch)
    styles = getSampleStyleSheet()

    title_s   = ParagraphStyle("T2",   parent=styles["Title"],   fontSize=20, spaceAfter=6)
    h1_s      = ParagraphStyle("H1",   parent=styles["Heading1"],fontSize=14, spaceAfter=4, spaceBefore=14)
    body_s    = ParagraphStyle("B2",   parent=styles["Normal"],  fontSize=11, spaceAfter=4, leading=16)
    bullet_s  = ParagraphStyle("Bul",  parent=styles["Normal"],  fontSize=11, spaceAfter=3, leftIndent=18, leading=15)
    meta_s    = ParagraphStyle("Meta", parent=styles["Normal"],  fontSize=9,  textColor=colors.grey, spaceAfter=2)
    ok_s      = ParagraphStyle("OK",   parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#1a7a1a"), spaceAfter=2)
    bad_s     = ParagraphStyle("Bad",  parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#cc0000"), spaceAfter=2)
    exp_s     = ParagraphStyle("Exp",  parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#555555"), leftIndent=18, spaceAfter=6)

    def HR():
        return HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#cccccc"), spaceAfter=6)

    story = []
    story.append(Paragraph(f"{title}: {profile['topic']}", title_s))
    story.append(Paragraph(
        f"Learner: {profile['name']}  |  Level: {profile['level']}  |  {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        meta_s))
    story.append(HR())

    story.append(Paragraph("Overview", h1_s))
    story.append(Paragraph(summary_data.get("summary",""), body_s))
    story.append(Spacer(1,6))

    story.append(Paragraph("Key Points", h1_s))
    for pt in summary_data.get("main_points",[]):
        story.append(Paragraph(f"• {pt}", bullet_s))
    story.append(Spacer(1,6))

    story.append(Paragraph("Important Questions to Review", h1_s))
    for i,kq in enumerate(summary_data.get("key_questions",[]),1):
        story.append(Paragraph(f"{i}. {kq}", bullet_s))

    # Mastery report extras
    if report_type == "mastery_report":
        score_data = (SESSION.get("diagnostic_score") or {})
        if score_data:
            story.append(HR())
            story.append(Paragraph("Assessment Results", h1_s))
            pct    = score_data.get("score_pct",0)*100
            passed = score_data.get("passed",False)
            story.append(Paragraph(
                f"Score: {score_data.get('correct',0)}/{score_data.get('total',0)} ({pct:.0f}%) — {'PASSED' if passed else 'NOT YET PASSED'}",
                body_s))

        gap_data = SESSION.get("gap_data")
        if gap_data and gap_data.get("gaps"):
            story.append(HR())
            story.append(Paragraph("Knowledge Gaps", h1_s))
            story.append(Paragraph(gap_data.get("summary",""), body_s))
            for g in gap_data.get("gaps",[]):
                story.append(Paragraph(f"• {g}", bullet_s))

            lp = gap_data.get("learning_path",[])
            if lp:
                story.append(HR())
                story.append(Paragraph("Your Personalised Learning Path", h1_s))
                for step in lp:
                    story.append(Paragraph(
                        f"{step['order']}. <b>{step['subtopic']}</b> — {step['why']}",
                        bullet_s))

    doc.build(story)
    print(f"  PDF exported: {filepath}")
    return str(filepath)


# ── 3. Quiz generation tool ──────────────────────────────────────────────────

def generate_quiz_questions(input_str: str) -> str:
    """
    Input: "topic|level|n_questions|missed_concepts"
      - missed_concepts is optional comma-separated list to focus on.
    Generates MCQs and returns them as a JSON string.
    """
    parts           = [p.strip() for p in input_str.split("|")]
    topic           = parts[0] if len(parts) > 0 else SESSION.get("profile", {}).get("topic", "Business Management")
    level           = parts[1] if len(parts) > 1 else "beginner"
    n               = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 5
    missed_raw      = parts[3] if len(parts) > 3 else ""
    missed          = [m.strip() for m in missed_raw.split(",")] if missed_raw else []

    ctx = SESSION.get("context") or retrieve_study_material(topic)
    print("***********************************************************************")
    print(topic, level, n, missed)
    print(ctx)
    print("***********************************************************************")

    difficulty_map = {
        "beginner":     "60% easy, 40% medium",
        "intermediate": "20% easy, 60% medium, 20% hard",
        "advanced":     "20% medium, 80% hard",
    }
    diff = difficulty_map.get(level, "40% easy, 60% medium")

    focus = f"\nFocus especially on these concepts the learner struggled with: {missed}\n" if missed else ""

    prompt = f"""You are a business management examiner.
    You must trictly stick to the stopic
Generate exactly {n} multiple choice questions about: {topic}
Learner level: {level} — difficulty mix: {diff}
{focus}
Mix question types: conceptual, scenario-based, definition, compare/contrast, true-or-false-as-MCQ.
Each question must have 4 options (A B C D) and exactly one correct answer.
Make questions detailed and comprehensive, testing deep understanding.
Base every question strictly on the context. No invented facts.

Context:
{ctx}

Return ONLY a valid JSON array. No markdown, no preamble.
Each item: {{"id":1,"type":"conceptual","difficulty":"easy","question":"...","options":{{"A":"...","B":"...","C":"...","D":"..."}},"correct_answer":"A","explanation":"one sentence why"}}"""

    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5,
        max_tokens=2000,
    )
    raw = (resp.choices[0].message.content or "").strip()

    # Clean fences
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(lines[1:])
        if raw.rstrip().endswith("```"):
            raw = raw.rstrip()[:-3]
    start = raw.find("["); end = raw.rfind("]")
    if start != -1 and end != -1:
        raw = raw[start:end+1]

    try:
        questions = json.loads(raw.strip())
        # Validate
        for q in questions:
            if q.get("correct_answer") not in q.get("options", {}):
                raise ValueError(f"correct_answer not in options: Q{q.get('id')}")
        SESSION["generated_quiz"] = questions
        return json.dumps(questions)
    except Exception as e:
        return json.dumps({"error": str(e), "raw": raw[:200]})




In [16]:
SESSION

{'profile': None,
 'context': '',
 'diagnostic_score': None,
 'gap_data': None,
 'adaptive_result': None,
 'summary': None}

In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# ── TOOL FUNCTIONS ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

# Note: The following functions are the tools that the agents will call to perform specific tasks like running the quiz, scoring it, analyzing gaps, etc. Each function takes a string input (which can be a JSON string or a delimited string), processes it, and returns a string output (often a JSON string) that the agent can use to decide the next steps.
# ── 4. Interactive quiz runner ────────────────────────────────────────────────

def run_interactive_quiz(dummy_input: str = "") -> str:
    """
    Runs the quiz interactively using questions stored in SESSION["generated_quiz"].
    No input parameter needed — reads from SESSION directly.
    Stores responses in SESSION["quiz_responses"].
    Returns a confirmation string.
    """
    questions = SESSION.get("generated_quiz")
    # return
    if not questions:
        return "ERROR: No quiz found in session. Call generate_quiz_questions first."
    # return questions
    if isinstance(questions, dict) and "error" in questions:
        return f"ERROR: Quiz has error: {questions['error']}"

    try:
        responses = []
        for q in questions:
            print(f"\n  [{q.get('difficulty','?').upper()}] [{q.get('type','MCQ')}] Q{q['id']}:")
            print(f"  {q['question']}\n")
            for letter, text in q["options"].items():
                print(f"    {letter}) {text}")

            while True:
                ans = input("\n  Your answer (A/B/C/D): ").strip().upper()
                if ans in ("A","B","C","D"):
                    break
                print("  Please enter A, B, C, or D.")

            correct = ans == q["correct_answer"]
            if correct:
                print("  ✓ Correct!")
            else:
                print("  ✗ Incorrect.")
                print(f"  Correct answer: {q['correct_answer']}) {q['options'][q['correct_answer']]}")
                print(f"  Why: {q.get('explanation','')}")

            responses.append({
                "question_id":    q["id"],
                "question":       q["question"],
                "difficulty":     q.get("difficulty","?"),
                "type":           q.get("type","MCQ"),
                "user_answer":    ans,
                "correct_answer": q["correct_answer"],
                "correct_option": q["options"][q["correct_answer"]],
                "explanation":    q.get("explanation",""),
                "is_correct":     correct,
            })
        
        SESSION["quiz_responses"] = responses
        return f"Quiz complete: {len(responses)} questions answered."
    except Exception as e:
        return f"ERROR during quiz: {str(e)}"


# ── 5. Score & save tool ─────────────────────────────────────────────────────

def score_and_save_quiz(input_str: str) -> str:
    """
    Input: "session_type|round_number"  e.g. "diagnostic|1" or "adaptive|2"
    Reads responses from SESSION["quiz_responses"], scores them, saves to DB.
    Returns score summary as JSON string.
    """
    parts        = input_str.split("|")
    session_type = parts[0].strip() if len(parts) > 0 else "quiz"
    round_num    = int(parts[1].strip()) if len(parts) > 1 and parts[1].strip().isdigit() else 1

    responses = SESSION.get("quiz_responses")
    if not responses:
        return json.dumps({"error": "No quiz responses found in session. Run run_interactive_quiz first."})

    total   = len(responses)
    correct = sum(1 for r in responses if r.get("is_correct"))
    pct     = correct / total if total else 0
    passed  = pct >= PASS_THRESHOLD
    gaps    = [r for r in responses if not r.get("is_correct")]

    score = {
        "total":            total,
        "correct":          correct,
        "score_pct":        round(pct, 3),
        "passed":           passed,
        "missed_concepts":  [r["question"][:80] for r in gaps],
        "gaps":             gaps,
    }

    profile = SESSION.get("profile")
    if profile:
        session_id = str(uuid.uuid4())[:8]
        conn = sqlite3.connect(DB_PATH)
        conn.execute(
            """INSERT INTO quiz_sessions
               (id,learner_id,session_type,topic,score_pct,passed,round_number,responses_json)
               VALUES (?,?,?,?,?,?,?,?)""",
            (session_id, profile["id"], session_type, profile["topic"],
             pct, int(passed), round_num, json.dumps(responses))
        )
        conn.commit(); conn.close()
    
    SESSION["diagnostic_score"] = score
    print(f"\n  Topic {profile['topic']}, Score: {correct}/{total} ({pct*100:.0f}%) — {'PASSED ✓' if passed else 'not yet passed'}")
    return json.dumps(score)


# ── 6. Gap analysis tool ──────────────────────────────────────────────────────

def analyse_knowledge_gaps(responses_json: str) -> str:
    """
    Input: JSON string of quiz responses (can be empty to read from SESSION["quiz_responses"]).
    Calls the LLM to identify knowledge gaps and produce an ordered learning path.
    Returns gap analysis as JSON string.
    """
    if responses_json.strip() == "" or responses_json.strip() == "{}":
        responses = SESSION.get("quiz_responses")
        if not responses:
            return json.dumps({"error": "No quiz responses found in session."})
    else:
        try:
            responses = json.loads(responses_json)
        except Exception:
            return json.dumps({"error": "Could not parse responses"})

    wrong = [r for r in responses if not r.get("is_correct")]
    if not wrong:
        result = {"gaps":[],"learning_path":[],"summary":"No gaps found — excellent work!"}
        SESSION["gap_data"] = result
        return json.dumps(result)

    profile = SESSION.get("profile", {"level":"beginner","topic":"Business Management"}) #We need to check this, this is not correct. It should not just plainly assume that the topic is about business managment
    wrong_qs = "\n".join(
        f"- Q: {r['question']} | Chose: {r['user_answer']} | Correct: {r['correct_answer']}) {r['correct_option']}"
        for r in wrong
    )

    prompt = f"""A {profile['level']} student studying '{profile['topic']}' answered these questions incorrectly:
{wrong_qs}

1. Identify the specific knowledge gaps (name the exact concepts not understood) thoroughly.
2. Create an ordered learning path of sub-topics to study, starting with foundational gaps.
3. Write one sentence explaining why each sub-topic matters.

Example:
{{"gaps":["Overfitting"],"learning_path": [{{"order": 1,"subtopic": "Training vs Validation Data","addresses_gap": ["Overfitting"],"why": "..."}},{{"order": 2,"subtopic": "Model Generalization","addresses_gap": ["Overfitting"],"why": "..."}}]}}

Return ONLY valid JSON — no markdown:
{{"gaps":["concept 1","concept 2"],"learning_path":[{{"order":1,"subtopic":"...","addresses_gap": "concept 1","why":"..."}}],"summary":"one sentence overall assessment"}}"""

    print()
    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3,
        # max_tokens=800
    )
    print(resp)
    raw = (resp.choices[0].message.content or "").strip()
    s   = raw[raw.find("{"):raw.rfind("}")+1]

    # s = '{"gaps":["Classification of proposed dividend in financial statements","Accounting for proposed dividend before AGM","Difference between proposed and declared dividends","Impact of AGM approval on proposed dividend","Disclosure requirements for proposed dividend under AS‑4"],"learning_path":[{"order":1,"subtopic":"Definition and nature of dividends in corporate finance","addresses_gap":["Classification of proposed dividend in financial statements"],"why":"Understanding what a dividend is and its purpose provides the context needed to classify it correctly in financial statements."},{"order":2,"subtopic":"Contingent liabilities vs. liabilities: accounting definitions and criteria","addresses_gap":["Classification of proposed dividend in financial statements","Difference between proposed and declared dividends"],"why":"Differentiating contingent liabilities from liabilities is essential to correctly classify proposed versus declared dividends."},{"order":3,"subtopic":"Accounting for proposed dividends: recognition criteria, measurement, and presentation before AGM","addresses_gap":["Accounting for proposed dividend before AGM","Classification of proposed dividend in financial statements"],"why":"Knowing when and how to record a proposed dividend ensures accurate financial reporting prior to shareholder approval."},{"order":4,"subtopic":"Accounting for declared dividends: recognition, measurement, and impact on equity and cash flows","addresses_gap":["Difference between proposed and declared dividends"],"why":"Clarifying the accounting treatment of declared dividends allows comparison with proposed dividends and understanding of subsequent financial effects."},{"order":5,"subtopic":"Effect of AGM approval: transition from proposed to declared dividend and its impact on financial statements","addresses_gap":["Impact of AGM approval on proposed dividend"],"why":"Recognizing the change at AGM is crucial for accurate liability recognition and equity adjustments."},{"order":6,"subtopic":"Disclosure requirements for proposed dividends under AS‑4: notes to accounts and regulatory expectations","addresses_gap":["Disclosure requirements for proposed dividend under AS‑4"],"why":"Compliance with disclosure standards ensures transparency and meets regulatory obligations."}],"summary":"The student needs to build a solid foundation on dividend concepts, liability classifications, and the specific accounting and disclosure rules governing proposed versus declared dividends."}'

    SESSION["gap_data_before_augment"] = s
    # return "Gap analysis result stored in session."
    # print(s)
    # print(json.loads(s))
    # return ""
    try:
        result = json.loads(s)
    except Exception as e:
        print(e)
        result = {
            "gaps":          [r["question"][:60] for r in wrong],
            "learning_path": [{"order":i+1,"subtopic":r["question"][:60],"why":"Answered incorrectly"}
                               for i,r in enumerate(wrong)],
            "summary":       f"{len(wrong)} concepts need review."
        }

    SESSION["gap_data"] = result

    # Save learning path JSON
    try:
        if SESSION.get("profile"):
            pf   = OUTPUT_DIR / f"learning_path_{SESSION['profile']['id']}_{SESSION['profile']['topic']}.json"
            with open(pf,"w") as f:
                json.dump({"profile":profile,"gap_data":result,"generated_at":datetime.now().isoformat()},f,indent=2)

            with open("./new_file.json", "w") as f:
                json.dump(json.loads(s), f, indent=2)
            print(f"  Learning path saved: {pf}")
    except Exception as e:
        print(f"  Could not save learning path: {str(e)}")

    # print(f"  Gap analysis: {result['summary']}")
    # print(f"  Gaps identified: {result['gaps']}")
    return json.dumps(result)


# ── 7. Check pass / loop control ─────────────────────────────────────────────

def check_pass_status(score_json: str) -> str:
    """
    Input: score JSON string from score_and_save_quiz.
    Returns a plain-English status the agent uses to decide whether to loop.
    """
    try:
        score = json.loads(score_json) if isinstance(score_json, str) else score_json
    except Exception:
        return "Could not parse score."
    pct    = score.get("score_pct", 0) * 100
    passed = score.get("passed", False)
    missed = score.get("missed_concepts", [])
    if passed:
        return f"PASSED with {pct:.0f}%. Proceed to mastery tracking."
    return (f"NOT PASSED — score {pct:.0f}%. "
            f"Missed concepts: {missed[:3]}. Generate new questions focusing on these gaps.")


# ── 8. Update mastery tool ───────────────────────────────────────────────────

def update_mastery_record(input_str: str) -> str:
    """
    Input: "score_pct|passed"  e.g. "0.85|True"
    Updates mastery record in DB and upgrades learner level if passed.
    Returns confirmation string.
    """
    parts  = input_str.split("|")
    pct    = float(parts[0].strip()) if len(parts) > 0 else 0.0
    passed = parts[1].strip().lower() == "true" if len(parts) > 1 else False

    profile = SESSION.get("profile")
    if not profile:
        return "No active profile in session."

    conn = sqlite3.connect(DB_PATH)
    existing = conn.execute(
        "SELECT best_score FROM mastery_records WHERE learner_id=? AND topic=?",
        (profile["id"], profile["topic"])
    ).fetchone()
    best = max((existing[0] if existing else 0), pct)
    if existing:
        conn.execute(
            "UPDATE mastery_records SET mastered=?,best_score=?,updated_at=datetime('now') WHERE learner_id=? AND topic=?",
            (int(passed), best, profile["id"], profile["topic"])
        )
    else:
        conn.execute(
            "INSERT INTO mastery_records (id,learner_id,topic,mastered,best_score) VALUES (?,?,?,?,?)",
            (str(uuid.uuid4())[:8], profile["id"], profile["topic"], int(passed), best)
        )
    if passed:
        conn.execute(
            "UPDATE learner_profiles SET level='intermediate' WHERE id=?",
            (profile["id"],)
        )
        SESSION["profile"]["level"] = "intermediate"
    conn.commit(); conn.close()

    status = "MASTERED" if passed else "IN PROGRESS"
    return f"Mastery record updated. Status: {status}. Best score: {best*100:.0f}%."



print("All tool functions defined.")


All tool functions defined.


In [180]:
SESSION


{'profile': None,
 'context': '',
 'diagnostic_score': None,
 'gap_data': None,
 'adaptive_result': None,
 'summary': None}

## Cell 5 — Wrap tools & build agents

Each agent gets:
- A set of `Tool` objects it can call
- A ReAct prompt (from LangChain hub) that tells it to reason → act → observe
- An `AgentExecutor` that runs the loop

In [19]:
# from langchain.agents import create_agent
# from langchain import hub
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

# # Pull the standard ReAct prompt once
# react_prompt = hub.pull("hwchase17/react")


In [20]:

# ═══════════════════════════════════════════════════════════════════════════
# ── WRAP FUNCTIONS AS LANGCHAIN TOOLS ──────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

# Wrap tools with StructuredTool for proper schema handling with Groq
tool_create_profile = StructuredTool.from_function(
    func=create_learner_profile,
    name="create_learner_profile",
    description=(
        "Create a new learner profile. "
        "Input must be a string in this format: 'name|topic|level'. "
        "Level must be beginner, intermediate, or advanced. "
        "Returns a JSON object with the profile including a learner ID."
    ),
)

tool_fetch_profile = StructuredTool.from_function(
    func=fetch_learner_profile,
    name="fetch_learner_profile",
    description="Fetch an existing learner profile by their learner ID string. Returns profile JSON.",
)

tool_retrieve = StructuredTool.from_function(
    func=retrieve_study_material,
    name="retrieve_study_material",
    description=(
        "Retrieve relevant study material from the knowledge base for a given topic. "
        "Input is the topic string. Returns context text used for quiz generation and summaries."
    ),
)

tool_generate_quiz = StructuredTool.from_function(
    func=generate_quiz_questions,
    name="generate_quiz_questions",
    description=(
        "Generate multiple choice quiz questions and store them in the session. "
        "Input format: 'topic|level|n_questions|missed_concepts' "
        "where missed_concepts is optional comma-separated concepts to focus on. "
        "Example: 'Financial Statements|beginner|5|balance sheet,accruals'. "
        "Stores quiz in SESSION and returns confirmation. Do NOT pass the returned JSON to run_interactive_quiz — just call run_interactive_quiz next."
    ),
)

tool_run_quiz = StructuredTool.from_function(
    func=run_interactive_quiz,
    name="run_interactive_quiz",
    description=(
        "Present quiz questions to the learner interactively. "
        "Reads questions from SESSION (populated by generate_quiz_questions). "
        "Input can be empty string or any value (ignored). "
        "Stores responses in SESSION and returns confirmation. "
        "Call this ONLY after generate_quiz_questions has been called."
    ),
)

tool_score = StructuredTool.from_function(
    func=score_and_save_quiz,
    name="score_and_save_quiz",
    description=(
        "Score quiz responses and save the session to the database. "
        "Input format: 'session_type|round_number' "
        "where session_type is 'diagnostic' or 'adaptive'. "
        "Example: 'diagnostic|1' or 'adaptive|2'. "
        "Reads responses from SESSION (populated by run_interactive_quiz). "
        "Returns score summary JSON with total, correct, score_pct, passed, and missed_concepts."
    ),
)

tool_analyse_gaps = StructuredTool.from_function(
    func=analyse_knowledge_gaps,
    name="analyse_knowledge_gaps",
    description=(
        "Analyse quiz responses to identify knowledge gaps and produce a personalised learning path. "
        "Input can be empty string to read from SESSION[\"quiz_responses\"], or a JSON string of responses. "
        "Returns gap analysis JSON with gaps list, ordered learning_path, and summary."
    ),
)

tool_check_pass = StructuredTool.from_function(
    func=check_pass_status,
    name="check_pass_status",
    description=(
        "Check whether the learner passed the quiz (80% threshold). "
        "Input is the score JSON string from score_and_save_quiz. "
        "Returns a plain-English status string: PASSED or NOT PASSED with guidance on next step."
    ),
)

tool_update_mastery = StructuredTool.from_function(
    func=update_mastery_record,
    name="update_mastery_record",
    description=(
        "Update the learner's mastery record in the database after completing the adaptive quiz. "
        "Input format: 'score_pct|passed' e.g. '0.85|True'. "
        "Returns a confirmation string."
    ),
)

tool_summary = StructuredTool.from_function(
    func=generate_study_summary,
    name="generate_study_summary",
    description=(
        "Generate a study summary for a topic including main points, overview, and key exam questions. "
        "Input is the topic string. "
        "Returns JSON with main_points, summary, and key_questions."
    ),
)

tool_export_pdf = StructuredTool.from_function(
    func=export_report_pdf,
    name="export_report_pdf",
    description=(
        "Export a PDF report for the current learner session. "
        "Input must be either 'quick_study' or 'mastery_report'. "
        "Returns the file path of the saved PDF."
    ),
)



In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# ── BUILD AGENTS ───────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

def make_agent(tools: list, system_note: str = ""):
    """Helper: build a ReAct AgentExecutor with the given tools."""
    # Create agent directly without bind_tools - StructuredTool handles schema correctly
    agent = create_agent(model=llm, tools=tools)
    return agent
    # return AgentExecutor.from_agent_and_tools(
    #     agent=agent,
    #     tools=tools,
    #     verbose=False,
    #     max_iterations=20,
    #     handle_parsing_errors=True,
    #     handle_tool_error=True,
    # )

# ── Knowledge Assessment Agent ───────────────────────────────────────────────
assessment_agent = make_agent(
    tools=[
        tool_retrieve, #(user queestion)
        tool_generate_quiz, #(generate quiz)
        tool_run_quiz, #adm
        tool_score, 
    ]
)

# ── Learning Path Agent ──────────────────────────────────────────────────────
learning_path_agent = make_agent(
    tools=[
        tool_analyse_gaps, # json 
    ]
)

# ── Adaptive Quiz Agent ──────────────────────────────────────────────────────
adaptive_quiz_agent = make_agent(
    tools=[
        tool_generate_quiz,
        tool_run_quiz,
        tool_score,
        tool_check_pass,
        tool_analyse_gaps,
    ]
)

# ── Mastery Tracking Agent ───────────────────────────────────────────────────
mastery_agent = make_agent(
    tools=[
        tool_update_mastery,
        tool_summary,
        tool_export_pdf,

    ]

)

print("  mastery_agent         — 3 tools")

print("  adaptive_quiz_agent   — 5 tools")

print("All agents ready.")

print("  learning_path_agent   — 1 tool")

print("  quick_study_agent     — 5 tools")

print("  assessment_agent      — 4 tools")


  mastery_agent         — 3 tools
  adaptive_quiz_agent   — 5 tools
All agents ready.
  learning_path_agent   — 1 tool
  quick_study_agent     — 5 tools
  assessment_agent      — 4 tools


In [161]:
# run_interactive_quiz("")

In [ ]:
# score_and_save_quiz("diagnostic|1")


  Topic Strengths of Balance Sheets, Score: 0/3 (0%) — not yet passed


'{"total": 3, "correct": 0, "score_pct": 0.0, "passed": false, "missed_concepts": ["What is the primary objective of financial statements?", "A potential investor is considering investing in a company and wants to assess i", "What are financial statements?"], "gaps": [{"question_id": 1, "question": "What is the primary objective of financial statements?", "difficulty": "easy", "type": "conceptual", "user_answer": "D", "correct_answer": "A", "correct_option": "To provide information about economic resources and obligations of a business", "explanation": "The primary objective of financial statements is to provide information about economic resources and obligations of a business to assist users in their decision-making process.", "is_correct": false}, {"question_id": 2, "question": "A potential investor is considering investing in a company and wants to assess its financial performance. Which of the following financial statements would be most useful to the investor?", "difficulty": "me

In [22]:
#Testing knowledge assessment agent with a sample question
# !pip install langgraph>=0.3.1
from json import tool
from langchain_core.tools import tool

from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
# from langgraph.prebuilt import ToolNode, create_react_agent, tools_condition
from langgraph.graph.message import add_messages


# import BaseMessage
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode

In [ ]:


# assessment_agent = make_agent(
#     tools=[
#         tool_retrieve, #(user queestion)
#         tool_generate_quiz, #(generate quiz)
#         tool_run_quiz, #adm
#         tool_score, 
#     ]
# )

# starting_system_prompt = "You are a helpful study assistant, that can help students to learn about any topic." \
# "" \
# "You can access a set of tools and agents such as:" \
# "AGENTS:" \
# "1. Assessment Agent: To test the learner's knowledge on the topic and identify weak areas." \
# "You MUST DO THE following in this order:" \
# "1. use the fetch_learner_profile tool to get the learner's profile and current level." \
# "1. Call the Assessment Agent to test the learner's knowledge." \



# @tool
# def assessment_agent_tool(name: str, level: str, topic: str) -> str:
#     """
#     Input: A question string from the user about the topic they are studying.
#     This tool will run the assessment agent to test the learner's knowledge on the topic and identify weak areas.
#     Returns a summary of the assessment results and identified gaps.
#     """

#     assessment_prompt = f"""
# You are a Knowledge Assessment Agent administering a diagnostic quiz.
# The learner is {name}, studying {topic} at {level} level.

# Your steps (in order):
# 1. Retrieve study material: use retrieve_study_material with input "{topic}"
# 2. Generate diagnostic questions: use generate_quiz_questions with input "{topic}|{level}|5"
# 3. Run the quiz: use run_interactive_quiz (it will read questions from SESSION)
# 4. Score the results: use score_and_save_quiz with input "diagnostic|1"

# The SESSION automatically carries quiz data between tools, so you don't need to pass JSON strings around.
# Complete all steps. The goal is to identify the learner's baseline knowledge.
# """
#     # Run the assessment agent with the user's question as input
#     result = assessment_agent.invoke({"input": assessment_prompt})
#     return result






In [ ]:
#Note: Test the generate_quiz_questions function 
# generate_quiz_questions("Financial Statements|advanced|3")

'[{"id": 1, "type": "conceptual", "difficulty": "medium", "question": "What is the primary purpose of a balance sheet in the context of financial statements?", "options": {"A": "To determine the operational results of an undertaking", "B": "To provide information required for decision-making by the management and other stakeholders", "C": "To show the changes in income, expenses, profits, and losses as a result of business operations during the year", "D": "To depict the financial position or status of an undertaking on a particular date"}, "correct_answer": "D", "explanation": "The balance sheet is a statement that shows all the assets owned by the concern, all the obligations or liabilities payable to outsiders or creditors, and claims of the owners on a particular date."}, {"id": 2, "type": "scenario-based", "difficulty": "hard", "question": "A company has issued 5,000, 10% debentures of Rs. 100 each at par but redeemable at a premium of 5% after 5 years. How would these items be sh

In [ ]:
#Note: Test the generate_study_summary function, it should provide a list of main points, a summary and some key questions based on the context that was retrieved for the topic "Financial Statements", like the one in the outputs pdf
# get_summary = tool_summary.func("Financial Statements")
# print(get_summary)

{"main_points":["Provide info on earning capacity", "Provide info on cash flows", "Help in decision-making for shareholders", "Aid trade associations and stock exchanges", "Disclose accounting policies", "Report activities affecting society", "Evaluate management effectiveness"], 
"summary":"Financial statements provide useful information about a company's economic resources, obligations, and earning capacity. They help investors, creditors, and shareholders make informed decisions and evaluate the company's financial performance. The primary objective of financial statements is to assist users in their decision-making by providing reliable and periodical information. Financial statements include the Statement of Profit and Loss and the Balance Sheet, which depict the financial position and operational results of a company.", 
"key_questions":["What are the primary objectives of financial statements?", "What information do financial statements provide to investors and creditors?", "How

In [ ]:


# ── Quick Study Agent ────────────────────────────────────────────────────────
quick_study_agent = make_agent(
    tools=[
        # tool_search_profile,
        tool_create_profile,
        tool_retrieve, #(balance)
        tool_generate_quiz,
        tool_summary,
        tool_export_pdf,
    ]
)

#Note: Create quick study agent
def run_quick_study(name: str, topic: str):
    """Supervisor routes here for mode 1."""
    print("\n" + "="*60)
    print("  QUICK STUDY AGENT — running")
    print("="*60)

    prompt = f"""
You are a Quick Study Agent. Your job:
1. Create a learner profile for {name} studying {topic} at intermediate level
2. Retrieve study material for {topic} using retrieve_study_material
Complete all steps in order. Be thorough.
3. Generate 5 quiz questions using generate_quiz_questions
   with input "{topic}|intermediate|5"
4. Generate a study summary using generate_study_summary with input "{topic}"
 5. Export a quick study PDF using export_report_pdf with input "quick_study"
"""

    result = quick_study_agent.invoke({"input": prompt})
    print("\nQuick Study complete.")
    print(f"Output: {result.get('output','')}")





## Cell 6 — Supervisor + main entry point

The supervisor is not itself a ReAct agent — it's a simple router that reads the user's
mode choice and hands off to the right agent chain. This keeps it predictable.
Inside each agent, the LLM decides which tools to call and in what order.

In [ ]:

run_full_scale("Francesca", "Types of Financial Statements", "beginner")


  FULL SCALE — Stage 1: Knowledge Assessment
  Profile created: {'id': '262e92a0', 'name': 'Francesca', 'topic': 'Reserve and Surplus', 'level': 'advanced'}
Retrieved 8 chunks for topic 'advanced'.Retrieved 8 chunks for topic 'Reserve and Surplus'.
Context preview:
ount.
viii) For a period of 5 years immediately proceeding the date at which
 balance sheet in prepared for:
(a)    Shares reserved under contracts/commitments.(b)   Number and class of shares bought back.
(c)    Number and class of shares allotted for consideration other
      than cash and bonus shares.
Reprint 2026-27
151 Financial Statements of a Company
ix) Terms of any securities convertible into equity/preference shares
issued along with earliest date of conversion in descending order,
st...
***********************************************************************
Reserve and Surplus intermediate 5 []
ount.
viii) For a period of 5 years immediately proceeding the date at which
 balance sheet in prepared for:
(a)    Sha

In [ ]:
# SESSION.get("quiz_responses")

[{'question_id': 1,
  'question': 'Which of the following is NOT required to be disclosed for each class of shares in the notes to accounts under the new disclosure requirements?',
  'difficulty': 'hard',
  'type': 'conceptual',
  'user_answer': 'A',
  'correct_answer': 'C',
  'correct_option': "The aggregate number of shares held by the company's ultimate holding company and its subsidiaries.",
  'explanation': 'The aggregate holdings of the ultimate holding company are disclosed separately, not per class of shares.',
  'is_correct': False},
 {'question_id': 2,
  'question': 'Based on the Dinkar Ltd. scenario, what should be the amount of share capital shown in the equity section of the balance sheet?',
  'difficulty': 'hard',
  'type': 'scenario-based',
  'user_answer': 'B',
  'correct_answer': 'A',
  'correct_option': '135,90,000',
  'explanation': 'After accounting for calls received, forfeited shares, and unpaid calls, the share capital totals 135,90,000.',
  'is_correct': False},

In [ ]:
#Note: Test Learning Path Agent with a sample prompt to analyse gaps and produce a learning path based on quiz responses stored in SESSION["quiz_responses"]. The prompt should instruct the agent to use the analyse_knowledge_gaps tool with an empty string input. The output is stored in json format in the outputs folder 
gap_prompt = f"""
You are a Learning Path Agent.
Analyse the learner's quiz responses to identify knowledge gaps.
Use analyse_knowledge_gaps with an empty string (it will read from SESSION["quiz_responses"]).
Produce an ordered learning path showing what the learner needs to study.
"""
# learning_path_agent.invoke({"input": gap_prompt})
learning_path_agent.invoke({"messages": [
        {
            "role": "user",
            "content": gap_prompt
        }
    ]})

  Learning path saved: outputs\learning_path_31f9cf57_Proposed Dividend.json
  Gap analysis: 5 concepts need review.
  Gaps identified: ['Which of the following is NOT required to be disclosed for e', 'Based on the Dinkar Ltd. scenario, what should be the amount', 'Subscribing but not fully paid-up capital refers to:', "Which statement best contrasts the presentation of 'Share ca", 'Which statement is true regarding the disclosure of shares h']


{'messages': [HumanMessage(content='\nYou are a Learning Path Agent.\nAnalyse the learner\'s quiz responses to identify knowledge gaps.\nUse analyse_knowledge_gaps with an empty string (it will read from SESSION["quiz_responses"]).\nProduce an ordered learning path showing what the learner needs to study.\n', additional_kwargs={}, response_metadata={}, id='1e6756ca-094e-4c64-a8c4-c6eac05ddec2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function analyse_knowledge_gaps with responses_json empty string.', 'tool_calls': [{'id': 'fc_be36b3ef-6723-4d5a-80b6-883dea727ca9', 'function': {'arguments': '{"responses_json":""}', 'name': 'analyse_knowledge_gaps'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 221, 'total_tokens': 267, 'completion_time': 0.048415739, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.010938865, 'prompt_tokens_details': None, 'queue_time': 0.045465914

In [23]:
from typing import Optional
#Note: No Langgraph yet, So far only two agents implemented
def run_full_scale(id: str, name: Optional[str], topic: Optional[str], level: Optional[str] = "intermediate"):
    
    """Supervisor routes here for mode 2. Runs 4 agents in sequence."""
    SESSION["context"] = ""
    # ── Stage 1: Create profile ──────────────────────────────────────────
    print("\n" + "="*60)
    print("  FULL SCALE — Stage 1: Knowledge Assessment")
    print("="*60)
    # profile_json = fetch_learner_profile(id) or create_learner_profile(f"{id}|{name}|{topic}|{level}")
    profile_json = create_learner_profile(f"{id}|{name}|{topic}|{level}")
    # return
    profile      = json.loads(profile_json)
    print(f"  Profile created: {profile}")

    # ── Stage 2: Knowledge Assessment Agent ─────────────────────────────
    assessment_prompt = f"""
You are a Knowledge Assessment Agent administering a diagnostic quiz.
The learner is {name}, studying {topic} at {level} level.

Your steps (in order):
1. Retrieve study material: use tool_retrieve with input "{topic}"
2. Generate diagnostic questions: use tool_generate_quiz_questions with input "{topic}|{level}|5"
3. Run the quiz: use tool_run_interactive_quiz (it will read questions from SESSION)
4. Score the results: use tool_score_and_save_quiz with input "diagnostic|1"

The SESSION automatically carries quiz data between tools, so you don't need to pass JSON strings around.
Complete all steps. The goal is to identify the learner's baseline knowledge.
"""
    assessment_result = assessment_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": assessment_prompt
        }
    ]
})
    print(f"\n  Assessment complete.")

    # Extract responses from session for gap analysis
    conn  = sqlite3.connect(DB_PATH)
    row   = conn.execute(
        "SELECT responses_json, score_pct, passed FROM quiz_sessions "
        "WHERE learner_id=? AND session_type='diagnostic' ORDER BY created_at DESC LIMIT 1",
        (profile["id"],)
    ).fetchone()
    conn.close()

    if not row:
        print("  Warning: could not retrieve diagnostic responses. Using empty gaps.")
        responses_json = "[]"
        score_pct, passed = 0.0, False
    else:
        responses_json, score_pct, passed = row[0], row[1], bool(row[2])

    print(f"  Diagnostic score: {score_pct*100:.0f}% — {'passed' if passed else 'not passed'}")


###############################################################
    # ── Stage 3: Learning Path Agent ────────────────────────────────────
    print("\n" + "-"*60)
    print("  FULL SCALE — Stage 3: Learning Path")
    print("-"*60)

    gap_prompt = f"""
You are a Learning Path Agent.
Analyse the learner's quiz responses to identify knowledge gaps.
Use analyse_knowledge_gaps with an empty string (it will read from SESSION["quiz_responses"]).
Produce an ordered learning path showing what the learner needs to study.
"""
    learning_path_agent.invoke({"messages": [
        {
            "role": "user",
            "content": gap_prompt
        }
    ]})
    print("  Learning path generated.")

#     # ── Stage 4: Adaptive Quiz Agent ────────────────────────────────────
#     print("\n" + "-"*60)
#     print("  FULL SCALE — Stage 4: Adaptive Quiz")
#     print("-"*60)
#     print(f"  Goal: reach {int(PASS_THRESHOLD*100)}% or above.")
#     print("  The agent will loop, changing questions each round, until you pass.\n")

#     gap_data     = SESSION.get("gap_data", {})
#     missed       = gap_data.get("gaps", [])
#     missed_str   = ", ".join(missed[:4]) if missed else ""

#     adaptive_prompt = f"""
# You are an Adaptive Quiz Agent. Your goal is to help {name} pass the quiz on {topic}
# by scoring {int(PASS_THRESHOLD*100)}% or higher.

# Rules:
# - Each round: generate_quiz_questions → run_interactive_quiz → score_and_save_quiz → check_pass_status
# - If NOT PASSED: loop and generate new questions focused on missed concepts
# - If PASSED: stop and report success
# - Maximum {MAX_QUIZ_ROUNDS} rounds
# - SESSION carries all data between tools automatically

# Focus areas: {missed_str if missed_str else 'general ' + topic}

# Start round 1 with input: "{topic}|{level}|5|{missed_str}"
# """
#     adaptive_result = adaptive_quiz_agent.invoke({"input": adaptive_prompt})
#     print("\n  Adaptive quiz complete.")

#     # ── Stage 5: Mastery Tracking Agent ─────────────────────────────────
#     print("\n" + "-"*60)
#     print("  FULL SCALE — Stage 5: Mastery Tracking & Report")
#     print("-"*60)

#     conn = sqlite3.connect(DB_PATH)
#     best_row = conn.execute(
#         "SELECT MAX(score_pct), MAX(passed) FROM quiz_sessions "
#         "WHERE learner_id=? AND session_type='adaptive'",
#         (profile["id"],)
#     ).fetchone()
#     conn.close()
#     best_score = best_row[0] if best_row and best_row[0] else score_pct
#     best_passed = bool(best_row[1]) if best_row and best_row[1] is not None else False

#     mastery_prompt = f"""
# You are a Mastery Tracking Agent. The learner {name} has completed the adaptive quiz on {topic}.
# Best score: {best_score*100:.0f}%. Passed: {best_passed}.

# Your steps (in order):
# 1. Update mastery record: use update_mastery_record with input "{best_score}|{best_passed}"
# 2. Generate study summary: use generate_study_summary with input "{topic}"
# 3. Export mastery report: use export_report_pdf with input "mastery_report"

# Complete all 3 steps.
# """
#     mastery_agent.invoke({"input": mastery_prompt})
#     print("\n  Mastery report complete.")


In [50]:

# ═══════════════════════════════════════════════════════════════════════════
# ── SUPERVISOR / ENTRY POINT ───────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

def run_tutor():
    print("\n" + "="*60)
    print("  ADAPTIVE BUSINESS MANAGEMENT TUTOR")
    print("  Multi-Agent System (LangChain ReAct)")
    print("="*60)

    name  = "Nandita"#input("\nYour name: ").strip() or "Learner"
    topic = "Finance"#input("Topic to study (e.g. Financial Statements, Marketing Mix): ").strip()
    if not topic:
        topic = "Business Management"

    print("\nStudy mode:")
    print("  1  Quick Study  — summary + quiz exported to PDF  (~2 min)")
    print("  2  Full Scale   — diagnostic → gaps → adaptive quiz → mastery report")
    mode = "1"#input("\nChoose (1 or 2): ").strip()

    if mode == "1":
        run_quick_study(name, topic)

    # elif mode == "2":
    #     print("\nExperience level:")
    #     print("  1  beginner  |  2  intermediate  |  3  advanced")
    #     lv = input("Choose (1/2/3) [1]: ").strip()
    #     level = {"1":"beginner","2":"intermediate","3":"advanced"}.get(lv, "beginner")
    #     run_full_scale(name, topic, level)

    # else:
    #     print("Invalid choice. Please enter 1 or 2.")

    print("\n" + "="*60)
    print("  Session complete. Check the outputs/ folder for your PDF.")
    print("="*60)



In [163]:
SESSION.keys()

dict_keys(['profile', 'context', 'diagnostic_score', 'gap_data', 'adaptive_result', 'summary', 'generated_quiz', 'quiz_responses', 'gap_data_before_augment'])

In [165]:
SESSION.get("gap_data")

{'gaps': ['Receivable classification',
  'Cash and cash equivalents definition (AS-3)',
  'Investment classification',
  'Provision classification'],
 'learning_path': [{'order': 1,
   'subtopic': 'Current vs Non‑Current classification of assets and liabilities',
   'addresses_gap': ['Receivable classification',
    'Investment classification',
    'Provision classification'],
   'why': 'It establishes the fundamental rule for determining liquidity and time horizon for all asset and liability categories.'},
  {'order': 2,
   'subtopic': 'Receivable classification based on maturity period',
   'addresses_gap': ['Receivable classification'],
   'why': 'It clarifies when a receivable should be treated as current or non‑current, directly affecting balance sheet presentation.'},
  {'order': 3,
   'subtopic': 'Investment classification based on expected realization period',
   'addresses_gap': ['Investment classification'],
   'why': 'It ensures investments are correctly reported as current 

In [ ]:

# ── START ──
run_tutor()

## Cell 7 — Inspect stored records (optional)

In [ ]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)

print("── Learner profiles ──")
df1 = pd.read_sql("SELECT * FROM learner_profiles ORDER BY created_at DESC LIMIT 10", conn)
print(df1.to_string(index=False) if not df1.empty else "  (none yet)")

print("\n── Quiz sessions ──")
df2 = pd.read_sql(
    "SELECT learner_id,session_type,topic,round_number,score_pct,passed,created_at "
    "FROM quiz_sessions ORDER BY created_at DESC LIMIT 20", conn
)
if not df2.empty:
    df2["score_pct"] = (df2["score_pct"]*100).round(0).astype(int).astype(str) + "%"
    print(df2.to_string(index=False))
else:
    print("  (none yet)")

print("\n── Mastery records ──")
df3 = pd.read_sql("SELECT * FROM mastery_records ORDER BY updated_at DESC", conn)
if not df3.empty:
    df3["best_score"] = (df3["best_score"]*100).round(0).astype(int).astype(str) + "%"
    print(df3.to_string(index=False))
else:
    print("  (none yet)")

conn.close()
